# Event-aligned portfolio impacts

Return-period flood depth is an exposure. An *impact function* maps that exposure
onto a decision quantity — here a hypothetical fractional damage ratio — while
preserving the same event alignment (the 100-year depth becomes the 100-year
damage ratio, not a re-ranked quantile).

1. **Raw evaluation** — depth at each return period (same API as the portfolio notebook).
2. **Impact evaluation** — `PiecewiseLinearImpact` from `crc_sdk.impacts` (backed by `crc_framework`).
3. **Dollar loss** — damage ratio × replacement value, compared before/after.

Builds on the same Cologne assets and canonical hazard file.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pyarrow.parquet as pq

from crc_sdk.connectors import read_hazard_dataset
from crc_sdk.impacts import PiecewiseLinearImpact
from crc_sdk.workflows import ExecutionOptions, HazardDataset

pio.renderers.default = "jupyterlab+png"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
FIXTURE_DIR = Path("../fixtures/os_climate")

HAZARD_NAME = "RiverineInundation"
PATHWAY = "historical"
HORIZON = 1980
RETURN_PERIODS = [25, 50, 100, 250, 500, 1000]

HAZARD_PATH = FIXTURE_DIR / "hazard.parquet"
ASSETS_PATH = FIXTURE_DIR / "assets.parquet"
RAW_PATH = DATA_DIR / "cologne_portfolio_before_impact.parquet"
IMPACT_PATH = DATA_DIR / "cologne_portfolio_after_impact.parquet"

DEPTH_DAMAGE = PiecewiseLinearImpact(
    exposure=[0.0, 0.1, 0.5, 1.0, 2.0, 4.0],
    impact=[0.0, 0.01, 0.08, 0.22, 0.55, 1.0],
)


## 1. Assets and hazard

Load the same checked-in Cologne assets and canonical hazard fixture used by the
portfolio notebook. No remote data access is required.


In [ ]:
assets = pq.read_table(ASSETS_PATH)
hazard = read_hazard_dataset(HAZARD_PATH)
print(f"assets: {assets.num_rows} <- {ASSETS_PATH}")
print(f"hazard: {hazard.num_rows} rows <- {HAZARD_PATH}")
assets.to_pandas()


## 2. Before / after impact

`.impact(...)` accepts any `ImpactFunction` — including framework builtins like
`PiecewiseLinearImpact` / `SigmoidImpact`, or a plain vectorized callable wrapped
as `CallableImpact`. The output keeps the same RP column names; only the value
semantics and units change (recorded in Parquet metadata as `event_aligned`).


In [ ]:
request = (
    HazardDataset.local(HAZARD_PATH)
    .for_assets(ASSETS_PATH)
    .select(
        hazard_names=[HAZARD_NAME],
        horizons=[HORIZON],
        pathways=[PATHWAY],
    )
    .return_periods(RETURN_PERIODS)
)
execution = ExecutionOptions(max_workers=1)
raw_result = request.write_parquet(RAW_PATH, execution=execution)
impact_result = request.impact(
    DEPTH_DAMAGE,
    name="piecewise_depth_damage",
    value_unit="fraction",
    value_semantics="hypothetical fractional asset damage",
).write_parquet(IMPACT_PATH, execution=execution)

raw = pq.read_table(RAW_PATH).to_pandas().set_index("asset_id")
impacted = pq.read_table(IMPACT_PATH).to_pandas().set_index("asset_id")
assert list(raw_result.value_columns) == list(impact_result.value_columns)

rows = []
for asset_id, raw_row in raw.iterrows():
    impact_row = impacted.loc[asset_id]
    replacement = float(raw_row["replacement_value"])
    for period, column in zip(RETURN_PERIODS, raw_result.value_columns):
        depth = float(raw_row[column])
        ratio = float(impact_row[column])
        rows.append(
            {
                "asset_id": asset_id,
                "return_period": period,
                "depth_m": depth,
                "damage_ratio": ratio,
                "replacement_value": replacement,
                "estimated_damage": ratio * replacement,
            }
        )
comparison = pd.DataFrame(rows)
comparison


## 3. Visualize


In [ ]:
# Depth–damage curve itself
curve_x = np.linspace(0, 4, 200)
curve_y = np.asarray(DEPTH_DAMAGE.evaluate(curve_x), dtype=float)
fig_curve = go.Figure(
    go.Scatter(
        x=curve_x,
        y=curve_y,
        mode="lines",
        line=dict(color="#2171B5", width=3),
        name="PiecewiseLinearImpact",
    )
)
fig_curve.add_trace(
    go.Scatter(
        x=DEPTH_DAMAGE.exposure,
        y=DEPTH_DAMAGE.impact,
        mode="markers",
        marker=dict(size=10, color="#D94801"),
        name="knots",
    )
)
fig_curve.update_layout(
    title="Hypothetical depth–damage curve",
    xaxis_title="Flood depth (m)",
    yaxis_title="Damage ratio",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig_curve.show()


In [ ]:
design = comparison[comparison["return_period"] == 100]
fig_damage = go.Figure(
    go.Bar(
        x=design["asset_id"],
        y=design["estimated_damage"],
        marker_color="#D94801",
        text=[f"${v:,.0f}" for v in design["estimated_damage"]],
        textposition="outside",
        customdata=np.stack(
            [design["depth_m"], design["damage_ratio"], design["replacement_value"]],
            axis=-1,
        ),
        hovertemplate=(
            "%{x}<br>damage=$%{y:,.0f}"
            "<br>depth=%{customdata[0]:.3f} m"
            "<br>ratio=%{customdata[1]:.3f}"
            "<br>replacement=$%{customdata[2]:,.0f}<extra></extra>"
        ),
    )
)
fig_damage.update_layout(
    title="Estimated damage at the 100-year return period",
    xaxis_title="Asset",
    yaxis_title="Estimated damage ($)",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig_damage.show()


In [ ]:
fig_escalation = go.Figure()
for asset_id, frame in comparison.groupby("asset_id"):
    frame = frame.sort_values("return_period")
    fig_escalation.add_trace(
        go.Scatter(
            x=[str(rp) for rp in frame["return_period"]],
            y=frame["estimated_damage"],
            mode="lines+markers",
            name=asset_id,
        )
    )
fig_escalation.update_layout(
    title="Estimated damage by return period",
    xaxis_title="Return period (years)",
    yaxis_title="Estimated damage ($)",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig_escalation.show()


## Scaling this up

[`pipelines/portfolio_impact_pipeline.py`](../pipelines/portfolio_impact_pipeline.py)
is the headless twin. It uses the same `PiecewiseLinearImpact` instance (picklable,
so `--max-workers` can rise above 1) and prints a depth / ratio / dollar table.
